# Angular 14 — Complete Reference Guide
### Detailed Notes | Examples | Use Cases | Interview Q&A

> **Angular 14** was released on **June 2, 2022**. It introduced **Standalone Components** (the most significant API evolution since Angular 2), **Strictly Typed Reactive Forms**, and multiple developer-experience improvements.

---

## Table of Contents
1. [Detailed Notes](#detailed-notes) — What changed and why
2. [Code Examples](#examples) — Hands-on code for every feature
3. [Use Cases](#use-cases) — Real-world application scenarios
4. [Interview Q&A](#interview-qa) — 20+ interview questions with answers

---

## Version Requirements

| Dependency | Required Version |
|---|---|
| Node.js | `^14.15.0 \|\| >=16.10.0` |
| TypeScript | `4.6.x` |
| RxJS | `^7.4.0` |
| Angular CLI | `14.x` |
| Zone.js | `~0.11.5` |

```bash
# Upgrade command
ng update @angular/core@14 @angular/cli@14
```

# Section 1 — Detailed Notes

---

## 1.1 Standalone Components (Developer Preview)

### What is a Standalone Component?
A **Standalone Component** is a component, directive, or pipe that does **not belong to any `NgModule`**. It declares its own dependencies directly via the `imports` array in its `@Component` decorator.

### Why Was This Introduced?
NgModules were one of Angular's most confusing concepts for beginners and caused boilerplate overhead:
- Every new component needed to be declared in an `NgModule`
- Sharing a component between modules required either re-declaring or exporting through a shared module
- Tree-shaking was harder because entire modules (not individual components) were imported

Standalone components eliminate the need for NgModules entirely.

### The `standalone: true` Flag
```typescript
@Component({
  selector: 'app-button',
  standalone: true,              // ← marks this as standalone
  imports: [CommonModule],       // ← import dependencies directly here
  template: `<button>{{ label }}</button>`
})
export class ButtonComponent {
  @Input() label = 'Click me';
}
```

### Three Types of Standalone
```typescript
// Standalone Component
@Component({ standalone: true, imports: [...], template: `...` })
export class MyComponent {}

// Standalone Directive
@Directive({ standalone: true, selector: '[myDirective]' })
export class MyDirective {}

// Standalone Pipe
@Pipe({ standalone: true, name: 'myPipe' })
export class MyPipe implements PipeTransform { transform(v: any) { return v; } }
```

### Bootstrapping a Standalone Application
```typescript
// main.ts — No AppModule at all!
import { bootstrapApplication } from '@angular/platform-browser';
import { AppComponent } from './app/app.component';

bootstrapApplication(AppComponent, {
  providers: [
    importProvidersFrom(RouterModule.forRoot(routes)),
    importProvidersFrom(HttpClientModule),
  ]
});
```

### Using Standalone Components inside NgModule Apps
```typescript
// Standalone components can be imported by NgModules
@NgModule({
  imports: [
    BrowserModule,
    ButtonComponent,   // ← import standalone component directly into NgModule
  ],
  declarations: [AppComponent],
  bootstrap: [AppComponent]
})
export class AppModule {}
```

### Lazy Loading with Standalone Components
```typescript
// routes.ts
const routes: Routes = [
  {
    path: 'settings',
    // Lazy load a SINGLE standalone component (no module needed!)
    loadComponent: () =>
      import('./settings/settings.component').then(m => m.SettingsComponent)
  },
  {
    path: 'admin',
    // Lazy load a set of routes (replaces lazy-loaded NgModule)
    loadChildren: () =>
      import('./admin/admin.routes').then(m => m.ADMIN_ROUTES)
  }
];
```

---

## 1.2 Strictly Typed Reactive Forms

### The Problem with Untyped Forms (Pre-Angular 14)
Before Angular 14, all reactive form controls returned `any`:
```typescript
const form = new FormGroup({ name: new FormControl(''), age: new FormControl(0) });
const name = form.get('name').value;   // type: any  ← no type safety!
form.get('nonExistent').value;         // compiles fine but runtime error!
```

### Typed Forms in Angular 14
```typescript
// Now fully generic — types are inferred automatically
const form = new FormGroup({
  name: new FormControl<string>(''),       // type: string | null
  age:  new FormControl<number>(0),        // type: number | null
  role: new FormControl<'admin' | 'user'>('user')
});

// Type-safe access
const name: string | null = form.controls.name.value;   // ✅ typed!
const age:  number | null = form.controls.age.value;    // ✅ typed!
form.controls.nonExistent;   // ❌ compile-time error — property doesn't exist
```

### `FormControl` Nullability
By default, `FormControl` allows `null` (when `.reset()` is called). To prevent `null`:
```typescript
// NonNullable — use { nonNullable: true }
const nameCtrl = new FormControl<string>('', { nonNullable: true });
nameCtrl.value;   // type: string  (no null!)

// Or use FormBuilder.nonNullable
const fb = inject(FormBuilder);
const form = fb.nonNullable.group({
  username: [''],    // string (non-nullable)
  password: ['']     // string (non-nullable)
});
```

### `FormArray` Typing
```typescript
// FormArray is typed with the element type
const tags = new FormArray<FormControl<string>>([
  new FormControl('angular', { nonNullable: true }),
  new FormControl('typescript', { nonNullable: true })
]);

tags.at(0).value;    // type: string ✅
tags.push(new FormControl<number>(1));  // ❌ compile error — wrong type!
```

### Gradual Migration with `UntypedFormControl`
```typescript
// Temporary escape hatch for existing code during migration
import { UntypedFormControl, UntypedFormGroup } from '@angular/forms';

const legacyForm = new UntypedFormGroup({
  name: new UntypedFormControl('')    // behaves exactly like old FormControl
});
```

---

## 1.3 Page Title Strategy

### Built-in Route Title Property
Angular 14 adds a `title` property to route configuration:
```typescript
const routes: Routes = [
  { path: '',       component: HomeComponent,    title: 'Home' },
  { path: 'about',  component: AboutComponent,   title: 'About Us' },
  { path: 'blog',   component: BlogComponent,    title: 'Blog' },
  { path: 'admin',  component: AdminComponent,   title: 'Admin Panel' }
];
```
The router automatically sets `document.title` to the matched route's `title`.

### Custom Title Strategy
Extend `TitleStrategy` to customize the title format:
```typescript
import { Injectable } from '@angular/core';
import { RouterStateSnapshot, TitleStrategy } from '@angular/router';

@Injectable({ providedIn: 'root' })
export class AppTitleStrategy extends TitleStrategy {
  override updateTitle(snapshot: RouterStateSnapshot): void {
    const title = this.buildTitle(snapshot);
    document.title = title ? `${title} | My App` : 'My App';
  }
}

// Register in providers:
{ provide: TitleStrategy, useClass: AppTitleStrategy }
```

### Dynamic Titles with Resolvers
```typescript
const routes: Routes = [
  {
    path: 'product/:id',
    component: ProductComponent,
    resolve: { product: productResolver },
    title: ProductTitleResolver   // can be a ResolveFn too
  }
];

// Functional title resolver
const ProductTitleResolver: ResolveFn<string> = (route) => {
  const productService = inject(ProductService);
  return productService.getProduct(route.paramMap.get('id')!).pipe(
    map(p => p.name)
  );
};
```

---

## 1.4 Extended Developer Diagnostics

New **compile-time checks** that warn about common template mistakes.

### `nullishCoalescingNotNullable`
Warns when `??` is used on a value that can't be null or undefined:
```html
<!-- ❌ Warning: user.name is always a string, ?? is unnecessary -->
<p>{{ user.name ?? 'Anonymous' }}</p>

<!-- ✅ Correct — only use ?? when value can actually be null/undefined -->
<p>{{ user.nickname ?? 'Anonymous' }}</p>
```

### `optionalChainNotNullable`
```html
<!-- ❌ Warning: items is declared as Item[], never null -->
<p>{{ items?.length }}</p>

<!-- ✅ Correct — only use ?. when property could be null -->
<p>{{ user?.profile?.avatar }}</p>
```

### Configuring Diagnostics
```json
// tsconfig.json
{
  "angularCompilerOptions": {
    "extendedDiagnostics": {
      "defaultCategory": "warning",
      "checks": {
        "nullishCoalescingNotNullable": "error",
        "optionalChainNotNullable": "warning",
        "invalidBananaInBox": "error"
      }
    }
  }
}
```

---

## 1.5 Optional Injectors in Embedded Views

`ViewContainerRef.createEmbeddedView()` and `createComponent()` now accept a custom `Injector`:

```typescript
const customInjector = Injector.create({
  parent: this.injector,
  providers: [
    { provide: MODAL_DATA, useValue: { userId: 42, mode: 'edit' } }
  ]
});

// Pass custom injector to provide scoped dependencies
const ref = this.vcr.createComponent(UserModalComponent, {
  injector: customInjector
});
```

---

## 1.6 `importProvidersFrom` Helper

Bridges the gap between NgModule-based providers and standalone apps:

```typescript
import { importProvidersFrom } from '@angular/core';

bootstrapApplication(AppComponent, {
  providers: [
    importProvidersFrom(RouterModule.forRoot(routes)),
    importProvidersFrom(HttpClientModule),
    importProvidersFrom(ReactiveFormsModule),
    importProvidersFrom(StoreModule.forRoot(reducers)),    // NgRx
    importProvidersFrom(EffectsModule.forRoot([AppEffects])),
    importProvidersFrom(TranslateModule.forRoot(translateConfig)),
  ]
});
```

---

## 1.7 TypeScript 4.6 Support

Key improvements Angular 14 benefits from:

### Destructured Discriminated Unions
```typescript
type Shape =
  | { kind: 'circle'; radius: number }
  | { kind: 'square'; side: number };

function area({ kind, ...rest }: Shape) {
  if (kind === 'circle') {
    // TypeScript 4.6 narrows 'rest' to { radius: number }
    return Math.PI * rest.radius ** 2;
  }
  return rest.side ** 2;  // narrowed to { side: number }
}
```

### Improved Recursion Checks
TypeScript 4.6 improves detection of infinite recursive types, which matters for Angular's typed forms inference chain.

# Section 2 — Code Examples

---

## Example 1: Full Standalone Application Setup

### Project Structure (No NgModule anywhere)
```
src/
├── main.ts                    ← bootstrapApplication()
├── app/
│   ├── app.component.ts       ← standalone: true, root component
│   ├── app.routes.ts          ← routes array (no RouterModule.forRoot)
│   ├── core/
│   │   ├── auth.guard.ts
│   │   └── auth.interceptor.ts
│   └── features/
│       ├── home/
│       │   └── home.component.ts   ← standalone: true
│       └── profile/
│           └── profile.component.ts ← standalone: true
```

```typescript
// app/app.routes.ts
import { Routes } from '@angular/router';

export const APP_ROUTES: Routes = [
  {
    path: '',
    loadComponent: () =>
      import('./features/home/home.component').then(m => m.HomeComponent),
    title: 'Home'
  },
  {
    path: 'profile',
    loadComponent: () =>
      import('./features/profile/profile.component').then(m => m.ProfileComponent),
    title: 'My Profile',
    canActivate: [() => inject(AuthService).isLoggedIn()]
  },
  {
    path: 'admin',
    loadChildren: () =>
      import('./features/admin/admin.routes').then(m => m.ADMIN_ROUTES),
    title: 'Admin'
  },
  { path: '**', redirectTo: '' }
];
```

```typescript
// main.ts
import { bootstrapApplication } from '@angular/platform-browser';
import { provideRouter, withPreloading, PreloadAllModules } from '@angular/router';
import { provideHttpClient, withInterceptors } from '@angular/common/http';
import { provideAnimations } from '@angular/platform-browser/animations';
import { importProvidersFrom } from '@angular/core';
import { APP_ROUTES } from './app/app.routes';
import { authInterceptor } from './app/core/auth.interceptor';
import { AppComponent } from './app/app.component';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(APP_ROUTES, withPreloading(PreloadAllModules)),
    provideHttpClient(withInterceptors([authInterceptor])),
    provideAnimations(),
    importProvidersFrom(TranslateModule.forRoot()),  // third-party NgModule bridge
  ]
}).catch(err => console.error(err));
```

```typescript
// app/app.component.ts
import { Component } from '@angular/core';
import { RouterOutlet, RouterLink, RouterLinkActive } from '@angular/router';
import { CommonModule } from '@angular/common';

@Component({
  selector: 'app-root',
  standalone: true,
  imports: [RouterOutlet, RouterLink, RouterLinkActive, CommonModule],
  template: `
    <nav>
      <a routerLink="/" routerLinkActive="active" [routerLinkActiveOptions]="{ exact: true }">Home</a>
      <a routerLink="/profile" routerLinkActive="active">Profile</a>
    </nav>
    <main>
      <router-outlet />
    </main>
  `
})
export class AppComponent {}
```

```typescript
// features/home/home.component.ts
import { Component, OnInit } from '@angular/core';
import { CommonModule } from '@angular/common';
import { HttpClient } from '@angular/common/http';

@Component({
  selector: 'app-home',
  standalone: true,
  imports: [CommonModule],   // only what THIS component needs
  template: `
    <h1>Welcome</h1>
    <ul>
      <li *ngFor="let post of posts">{{ post.title }}</li>
    </ul>
  `
})
export class HomeComponent implements OnInit {
  posts: any[] = [];

  constructor(private http: HttpClient) {}

  ngOnInit() {
    this.http.get<any[]>('https://jsonplaceholder.typicode.com/posts?_limit=5')
      .subscribe(data => this.posts = data);
  }
}
```

---

## Example 2: Strictly Typed Reactive Forms — Registration Form

```typescript
// registration.component.ts
import { Component } from '@angular/core';
import { CommonModule } from '@angular/common';
import { ReactiveFormsModule, FormBuilder, FormGroup, FormControl, Validators } from '@angular/forms';

// Define the shape of the form
interface RegistrationForm {
  firstName:   FormControl<string>;
  lastName:    FormControl<string>;
  email:       FormControl<string>;
  password:    FormControl<string>;
  age:         FormControl<number | null>;
  role:        FormControl<'user' | 'admin'>;
  acceptTerms: FormControl<boolean>;
}

@Component({
  selector: 'app-registration',
  standalone: true,
  imports: [CommonModule, ReactiveFormsModule],
  template: `
    <form [formGroup]="form" (ngSubmit)="onSubmit()">
      <input formControlName="firstName" placeholder="First Name">
      <span *ngIf="form.controls.firstName.invalid && form.controls.firstName.touched">
        First name is required
      </span>

      <input formControlName="email" type="email" placeholder="Email">
      <input formControlName="password" type="password" placeholder="Password">

      <input formControlName="age" type="number" placeholder="Age">

      <select formControlName="role">
        <option value="user">User</option>
        <option value="admin">Admin</option>
      </select>

      <input formControlName="acceptTerms" type="checkbox">

      <button type="submit" [disabled]="form.invalid">Register</button>
    </form>
  `
})
export class RegistrationComponent {
  private fb = inject(FormBuilder);

  // Fully typed form using FormGroup generic
  form = this.fb.nonNullable.group<RegistrationForm>({
    firstName:   this.fb.nonNullable.control('', Validators.required),
    lastName:    this.fb.nonNullable.control('', Validators.required),
    email:       this.fb.nonNullable.control('', [Validators.required, Validators.email]),
    password:    this.fb.nonNullable.control('', [Validators.required, Validators.minLength(8)]),
    age:         new FormControl<number | null>(null),
    role:        this.fb.nonNullable.control<'user' | 'admin'>('user'),
    acceptTerms: this.fb.nonNullable.control(false, Validators.requiredTrue)
  });

  onSubmit() {
    if (this.form.valid) {
      // All values are fully typed — no casting needed!
      const { firstName, email, role } = this.form.getRawValue();
      // firstName: string
      // email:     string
      // role:      'user' | 'admin'
      console.log('Submitting:', { firstName, email, role });
    }
  }
}
```

---

## Example 3: Typed FormArray — Dynamic Skills List

```typescript
// skills-form.component.ts
import { Component } from '@angular/core';
import { CommonModule } from '@angular/common';
import { ReactiveFormsModule, FormArray, FormControl, FormGroup, Validators } from '@angular/forms';

interface SkillControl {
  name:  FormControl<string>;
  level: FormControl<'beginner' | 'intermediate' | 'expert'>;
  years: FormControl<number>;
}

@Component({
  selector: 'app-skills-form',
  standalone: true,
  imports: [CommonModule, ReactiveFormsModule],
  template: `
    <div formArrayName="skills">
      <div *ngFor="let skill of skills.controls; let i = index" [formGroupName]="i">
        <input formControlName="name" placeholder="Skill name">
        <select formControlName="level">
          <option value="beginner">Beginner</option>
          <option value="intermediate">Intermediate</option>
          <option value="expert">Expert</option>
        </select>
        <input formControlName="years" type="number" placeholder="Years">
        <button type="button" (click)="removeSkill(i)">Remove</button>
      </div>
    </div>
    <button type="button" (click)="addSkill()">Add Skill</button>
  `
})
export class SkillsFormComponent {
  // Typed FormArray<FormGroup<SkillControl>>
  skills = new FormArray<FormGroup<SkillControl>>([]);

  addSkill() {
    this.skills.push(
      new FormGroup<SkillControl>({
        name:  new FormControl<string>('', { nonNullable: true, validators: Validators.required }),
        level: new FormControl<'beginner' | 'intermediate' | 'expert'>('beginner', { nonNullable: true }),
        years: new FormControl<number>(0, { nonNullable: true })
      })
    );
  }

  removeSkill(index: number) {
    this.skills.removeAt(index);
  }

  getSkillValues() {
    return this.skills.controls.map(group => group.getRawValue());
    // Returns: Array<{ name: string; level: 'beginner'|'intermediate'|'expert'; years: number }>
    // Fully typed — no casting!
  }
}
```

---

## Example 4: Page Title Strategy — Blog Application

```typescript
// app/core/title.strategy.ts
import { Injectable, inject } from '@angular/core';
import { RouterStateSnapshot, TitleStrategy } from '@angular/router';
import { Title } from '@angular/platform-browser';

@Injectable({ providedIn: 'root' })
export class BlogTitleStrategy extends TitleStrategy {
  private title = inject(Title);

  override updateTitle(snapshot: RouterStateSnapshot): void {
    const routeTitle = this.buildTitle(snapshot);
    if (routeTitle) {
      this.title.setTitle(`${routeTitle} — Angular Blog`);
    } else {
      this.title.setTitle('Angular Blog');
    }
  }
}
```

```typescript
// app/app.routes.ts
import { ResolveFn } from '@angular/router';

// Dynamic title resolver for blog post page
const postTitleResolver: ResolveFn<string> = (route) => {
  const postService = inject(PostService);
  const slug = route.paramMap.get('slug')!;
  return postService.getPost(slug).pipe(map(post => post.title));
};

export const APP_ROUTES: Routes = [
  { path: '',          loadComponent: () => import('./home/home.component').then(m => m.HomeComponent),   title: 'Latest Posts' },
  { path: 'about',     loadComponent: () => import('./about/about.component').then(m => m.AboutComponent), title: 'About' },
  { path: 'contact',   loadComponent: () => import('./contact/contact.component').then(m => m.ContactComponent), title: 'Contact Us' },
  {
    path: 'post/:slug',
    loadComponent: () => import('./post/post.component').then(m => m.PostComponent),
    title: postTitleResolver    // dynamic title from resolver
  },
];
```

```typescript
// main.ts — register custom strategy
bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(APP_ROUTES),
    { provide: TitleStrategy, useClass: BlogTitleStrategy }
  ]
});

// Result — browser tab titles:
// /           → "Latest Posts — Angular Blog"
// /about      → "About — Angular Blog"
// /post/ivy   → "Understanding Ivy — Angular Blog"  (from resolver)
```

---

## Example 5: Extended Developer Diagnostics Setup

```json
// tsconfig.json
{
  "compilerOptions": { "strict": true },
  "angularCompilerOptions": {
    "strictTemplates": true,
    "extendedDiagnostics": {
      "defaultCategory": "warning",
      "checks": {
        "nullishCoalescingNotNullable": "error",
        "optionalChainNotNullable": "warning",
        "invalidBananaInBox": "error",
        "missingControlFlowDirective": "warning",
        "textAttributeNotBinding": "warning"
      }
    }
  }
}
```

```typescript
// user.component.ts — demonstrating diagnostic catches
@Component({
  standalone: true,
  imports: [CommonModule],
  template: `
    <!-- ❌ DIAGNOSTIC ERROR: name is string, never null — ?? is pointless -->
    <p>{{ user.name ?? 'Unknown' }}</p>

    <!-- ✅ CORRECT: nickname can be null -->
    <p>{{ user.nickname ?? 'No nickname' }}</p>

    <!-- ❌ DIAGNOSTIC WARNING: items: Item[] — ?. is unnecessary -->
    <p>Total: {{ items?.length }}</p>

    <!-- ✅ CORRECT: optionalItems can be undefined -->
    <p>Total: {{ optionalItems?.length }}</p>

    <!-- ❌ DIAGNOSTIC ERROR: wrong two-way binding syntax -->
    <input ([ngModel])="value">

    <!-- ✅ CORRECT: banana-in-a-box syntax -->
    <input [(ngModel)]="value">
  `
})
export class UserComponent {
  user: { name: string; nickname: string | null } = { name: 'Alice', nickname: null };
  items: string[] = ['a', 'b'];
  optionalItems?: string[];
  value = '';
}
```

---

## Example 6: Optional Injectors — Scoped Modal with Data

```typescript
// modal.tokens.ts
import { InjectionToken } from '@angular/core';

export interface ModalConfig<T = unknown> {
  data: T;
  closeable: boolean;
  title: string;
}

export const MODAL_CONFIG = new InjectionToken<ModalConfig>('MODAL_CONFIG');
```

```typescript
// modal.service.ts
@Injectable({ providedIn: 'root' })
export class ModalService {
  constructor(
    private vcr: ViewContainerRef,
    private injector: Injector
  ) {}

  open<T>(component: Type<any>, config: ModalConfig<T>): ComponentRef<any> {
    // Create scoped injector with modal-specific data
    const modalInjector = Injector.create({
      parent: this.injector,
      providers: [
        { provide: MODAL_CONFIG, useValue: config }
      ]
    });

    // Pass the custom injector — Angular 14 feature
    const ref = this.vcr.createComponent(component, {
      injector: modalInjector
    });

    return ref;
  }
}
```

```typescript
// user-edit-modal.component.ts
@Component({
  standalone: true,
  imports: [CommonModule, ReactiveFormsModule],
  template: `
    <div class="modal">
      <h2>{{ config.title }}</h2>
      <form [formGroup]="form" (ngSubmit)="save()">
        <input formControlName="name">
        <button type="submit">Save</button>
      </form>
    </div>
  `
})
export class UserEditModalComponent {
  // Inject the scoped modal config
  config = inject<ModalConfig<User>>(MODAL_CONFIG);

  form = new FormGroup({
    name: new FormControl<string>(this.config.data.name, { nonNullable: true })
  });

  save() {
    console.log('Saved:', this.form.getRawValue());
  }
}

// Usage:
this.modalService.open(UserEditModalComponent, {
  title: 'Edit User',
  closeable: true,
  data: { id: 1, name: 'Alice', email: 'alice@example.com' }
});
```

---

## Example 7: Migration — NgModule App to Standalone (Step by Step)

```typescript
// STEP 1 — Run automatic migration schematic
// ng generate @angular/core:standalone

// STEP 2 — Before (NgModule-based)
// app.module.ts
@NgModule({
  declarations: [AppComponent, HomeComponent, HeaderComponent],
  imports: [BrowserModule, CommonModule, RouterModule.forRoot(routes)],
  providers: [UserService],
  bootstrap: [AppComponent]
})
export class AppModule {}

// STEP 3 — After (Standalone)
// Each component gets standalone: true + imports its own dependencies
// AppModule is deleted entirely

// app.component.ts (after migration)
@Component({
  selector: 'app-root',
  standalone: true,
  imports: [RouterOutlet, HeaderComponent],   // explicit imports
  template: `<app-header /><router-outlet />`
})
export class AppComponent {}

// main.ts (after migration)
bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes),
    UserService   // moved from AppModule.providers
  ]
});
```

# Section 3 — Use Cases

---

## Use Case 1: Micro-Frontend Architecture with Standalone Components

**Problem:** A large banking portal has multiple teams working on different sections (Accounts, Payments, Loans, Analytics). They need independent deployments and codebases but share UI components.

**Solution with Angular 14 Standalone Components:**

```typescript
// shared-ui library — npm package published by the design team
// @bank/shared-ui/button.component.ts
@Component({
  selector: 'bank-button',
  standalone: true,
  imports: [CommonModule],
  template: `
    <button [class]="variant" [disabled]="disabled" (click)="clicked.emit()">
      <ng-content></ng-content>
    </button>
  `
})
export class BankButtonComponent {
  @Input() variant: 'primary' | 'secondary' | 'danger' = 'primary';
  @Input() disabled = false;
  @Output() clicked = new EventEmitter<void>();
}
```

```typescript
// Accounts team — Angular 14 standalone micro-app
// accounts/main.ts
bootstrapApplication(AccountsAppComponent, {
  providers: [
    provideRouter(accountsRoutes),
    provideHttpClient(),
    importProvidersFrom(AccountsStoreModule)
  ]
});

// accounts/transfer/transfer.component.ts
@Component({
  standalone: true,
  imports: [
    CommonModule,
    ReactiveFormsModule,
    BankButtonComponent      // ← import from shared-ui directly, no SharedModule!
  ],
  template: `
    <form [formGroup]="transferForm" (ngSubmit)="submit()">
      <input formControlName="amount" type="number">
      <bank-button variant="primary" (clicked)="submit()">Transfer</bank-button>
      <bank-button variant="secondary" (clicked)="cancel()">Cancel</bank-button>
    </form>
  `
})
export class TransferComponent {
  transferForm = new FormGroup({
    amount: new FormControl<number>(0, { nonNullable: true }),
    toAccount: new FormControl<string>('', { nonNullable: true })
  });

  submit() {
    const { amount, toAccount } = this.transferForm.getRawValue();
    // amount: number — fully typed ✅
  }
}
```

**Benefits:**
- Shared UI components consumed without `SharedModule` boilerplate
- Each micro-app bootstraps independently
- Type-safe forms prevent incorrect data submission

---

## Use Case 2: Form-Heavy CRM Application — Typed Reactive Forms

**Problem:** A CRM app has 30+ forms for leads, contacts, deals, and activities. Previously using `any`-typed forms caused:
- Runtime errors when accessing `form.get('wrongField').value`
- Bugs where numeric fields were submitted as strings
- No autocomplete support in IDEs

**Solution with Angular 14 Typed Reactive Forms:**

```typescript
// lead.types.ts — define form types once, reuse everywhere
export interface LeadFormValue {
  firstName:   string;
  lastName:    string;
  email:       string;
  phone:       string | null;
  company:     string;
  dealValue:   number;
  stage:       'prospect' | 'qualified' | 'proposal' | 'closed';
  assignedTo:  number;           // user ID
  tags:        string[];
  notes:       string | null;
}
```

```typescript
// lead-form.service.ts — factory service for typed form
import { Injectable } from '@angular/core';
import { FormBuilder, FormControl, FormGroup } from '@angular/forms';
import { LeadFormValue } from './lead.types';

type LeadFormControls = {
  [K in keyof LeadFormValue]: FormControl<LeadFormValue[K]>
};

@Injectable({ providedIn: 'root' })
export class LeadFormService {
  private fb = inject(FormBuilder);

  createLeadForm(initial?: Partial<LeadFormValue>): FormGroup<LeadFormControls> {
    return this.fb.group<LeadFormControls>({
      firstName:   new FormControl<string>(initial?.firstName ?? '', { nonNullable: true }),
      lastName:    new FormControl<string>(initial?.lastName ?? '', { nonNullable: true }),
      email:       new FormControl<string>(initial?.email ?? '', { nonNullable: true }),
      phone:       new FormControl<string | null>(initial?.phone ?? null),
      company:     new FormControl<string>(initial?.company ?? '', { nonNullable: true }),
      dealValue:   new FormControl<number>(initial?.dealValue ?? 0, { nonNullable: true }),
      stage:       new FormControl<LeadFormValue['stage']>('prospect', { nonNullable: true }),
      assignedTo:  new FormControl<number>(0, { nonNullable: true }),
      tags:        new FormControl<string[]>([], { nonNullable: true }),
      notes:       new FormControl<string | null>(null),
    });
  }
}
```

```typescript
// lead-edit.component.ts
@Component({ standalone: true, imports: [ReactiveFormsModule, CommonModule] })
export class LeadEditComponent implements OnInit {
  private leadFormService = inject(LeadFormService);
  private leadService = inject(LeadService);

  form!: ReturnType<LeadFormService['createLeadForm']>;

  async ngOnInit() {
    const lead = await firstValueFrom(this.leadService.getLead(this.leadId));
    this.form = this.leadFormService.createLeadForm(lead);
  }

  onSubmit() {
    const value = this.form.getRawValue();
    // value.dealValue: number  ✅ — not 'any', not string
    // value.stage:     'prospect' | 'qualified' | 'proposal' | 'closed'  ✅
    this.leadService.updateLead(this.leadId, value);
  }
}
```

---

## Use Case 3: SEO-Optimized Blog — Page Title Strategy

**Problem:** A tech blog built in Angular has poor SEO because `document.title` was always "My Blog" regardless of which article was being read. Google couldn't index individual articles properly.

**Solution with Angular 14 Page Title + Dynamic Resolver:**

```typescript
// blog-title.strategy.ts
@Injectable({ providedIn: 'root' })
export class BlogTitleStrategy extends TitleStrategy {
  private readonly siteName = 'Angular Dev Blog';
  private title = inject(Title);
  private meta = inject(Meta);

  override updateTitle(snapshot: RouterStateSnapshot): void {
    const routeTitle = this.buildTitle(snapshot);
    const fullTitle = routeTitle ? `${routeTitle} | ${this.siteName}` : this.siteName;

    this.title.setTitle(fullTitle);

    // Also update Open Graph meta tag for social sharing
    this.meta.updateTag({ property: 'og:title', content: fullTitle });
  }
}
```

```typescript
// blog routes with dynamic title resolvers
export const BLOG_ROUTES: Routes = [
  {
    path: '',
    component: BlogListComponent,
    title: 'Latest Articles'
    // → browser tab: "Latest Articles | Angular Dev Blog"
  },
  {
    path: 'category/:slug',
    component: CategoryComponent,
    resolve: { category: categoryResolver },
    title: (route, state) => {
      const category = route.data['category'] as Category;
      return `${category.name} Articles`;
    }
    // → browser tab: "TypeScript Articles | Angular Dev Blog"
  },
  {
    path: 'post/:slug',
    component: PostComponent,
    resolve: { post: postResolver },
    title: postTitleResolver
    // → browser tab: "Understanding Signals in Angular | Angular Dev Blog"
  }
];

const postTitleResolver: ResolveFn<string> = (route) => {
  return inject(PostService).getPost(route.paramMap.get('slug')!).pipe(
    map(post => post.title)
  );
};
```

**Impact:**
- Individual article pages now indexed by Google with correct titles
- Social sharing previews show article titles
- Browser history shows meaningful page names

---

## Use Case 4: Design System Library — Standalone Components as NPM Package

**Problem:** A company's internal UI library (`@company/ui`) was published as an NgModule. Every consumer had to import `UiModule` which included ALL 80+ components, even if they only needed 3.

**Solution with Angular 14 Standalone Components:**

```typescript
// @company/ui — published as standalone components
// Each component is its own import:

// packages/ui/src/button/button.component.ts
@Component({
  selector: 'co-button',
  standalone: true,
  template: `<button class="co-btn co-btn--{{variant}}"><ng-content/></button>`
})
export class CoButtonComponent {
  @Input() variant: 'primary' | 'ghost' = 'primary';
}

// packages/ui/src/index.ts — public API
export { CoButtonComponent } from './button/button.component';
export { CoInputComponent }  from './input/input.component';
export { CoModalComponent }  from './modal/modal.component';
// ... 80 more individual exports
```

```typescript
// Consumer app — only import what you need!
@Component({
  standalone: true,
  imports: [
    CoButtonComponent,   // ← just this one, tree-shaker removes the rest!
    ReactiveFormsModule
  ],
  template: `<co-button variant="primary">Submit</co-button>`
})
export class CheckoutComponent {}

// Bundle size comparison:
// Before (NgModule UiModule import): +340KB (all 80 components)
// After  (Standalone imports):       +4KB   (only CoButtonComponent)
```

---

## Use Case 5: Code Quality — Diagnostics Catching Template Bugs Early

**Problem:** A fintech dashboard had a subtle bug: `account.balance ?? 0` was used throughout templates, but `balance` was typed as `number` (never null). The `??` was masking a bug where `balance` was actually `undefined` due to a missing API field — the `?? 0` silently fixed it at runtime but the root cause was never found.

**Solution: Enable Extended Diagnostics in Angular 14**

```json
// tsconfig.json — enable strict diagnostics
{
  "angularCompilerOptions": {
    "extendedDiagnostics": {
      "checks": {
        "nullishCoalescingNotNullable": "error"
      }
    }
  }
}
```

```html
<!-- Now Angular build FAILS at compile time: -->
<!-- ERROR: balance is declared as 'number', which is not nullable -->
<p>{{ account.balance ?? 0 }}</p>

<!-- Team must fix the ROOT CAUSE: update the type or fix the API -->
```

```typescript
// Fix: update the interface to reflect reality
interface Account {
  id: number;
  balance: number | null;   // was: number (incorrect)
}

// Now ?? is valid and the code is honest about nullability
<p>{{ account.balance ?? 0 }}</p>   // ✅ balance can be null — ?? is correct
```

---

## Use Cases Summary Table

| Use Case | Feature Used | Business Value |
|---|---|---|
| Banking Micro-Frontend | Standalone components | Independent team deployments, no SharedModule overhead |
| CRM Forms | Typed Reactive Forms | Zero runtime type errors, IDE autocomplete on form values |
| SEO Blog | Page Title Strategy | Better Google indexing, social sharing previews |
| UI Design System | Standalone as npm package | 98% smaller consumer bundle (tree-shakable) |
| Fintech Dashboard | Extended Diagnostics | Catch template bugs at build time, not in production |

# Section 4 — Interview Q&A

---

## Basic Level Questions

---

### Q1. What is a Standalone Component in Angular 14?

**Answer:**
A **Standalone Component** is a component, directive, or pipe that declares its own dependencies directly in its `@Component` decorator using the `imports` array, without needing to belong to an `NgModule`. It is marked with `standalone: true`.

```typescript
@Component({
  selector: 'app-card',
  standalone: true,              // no NgModule needed
  imports: [CommonModule],       // dependencies imported here
  template: `<div>{{ title }}</div>`
})
export class CardComponent {
  @Input() title = '';
}
```

Key benefits: less boilerplate, better tree-shaking, cleaner lazy loading, easier to test.

---

### Q2. What problem do Typed Reactive Forms solve?

**Answer:**
Before Angular 14, all reactive form controls had the type `any`, causing:
- No compile-time validation when accessing wrong control names
- No type inference on `.value` (returns `any`)
- Runtime errors only discovered in production

Angular 14 makes forms fully generic. `FormControl<string>` now returns `string | null`, and accessing a non-existent control is a **compile-time error**:

```typescript
const form = new FormGroup({ name: new FormControl<string>('') });
form.controls.name.value;      // ✅ type: string | null
form.controls.missing.value;   // ❌ compile error — 'missing' doesn't exist
```

---

### Q3. What is `bootstrapApplication` in Angular 14?

**Answer:**
`bootstrapApplication` is a function that bootstraps a **standalone component** as the root of an Angular application, **without any `NgModule`**:

```typescript
import { bootstrapApplication } from '@angular/platform-browser';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes),
    provideHttpClient()
  ]
});
```

It replaces `platformBrowserDynamic().bootstrapModule(AppModule)`. The second argument accepts providers to configure the application.

---

### Q4. What is `importProvidersFrom` and when do you use it?

**Answer:**
`importProvidersFrom` is a helper that **extracts providers from an `NgModule`** so they can be used in a standalone app's `bootstrapApplication` providers:

```typescript
bootstrapApplication(AppComponent, {
  providers: [
    importProvidersFrom(RouterModule.forRoot(routes)),     // Angular router
    importProvidersFrom(StoreModule.forRoot(reducers)),    // NgRx store
    importProvidersFrom(TranslateModule.forRoot(config)),  // ngx-translate
  ]
});
```

It is a **bridge** between the NgModule world and standalone apps — used when a third-party library hasn't migrated to provide standalone-compatible functions yet.

---

### Q5. What is the `title` property in Angular 14 route configuration?

**Answer:**
Angular 14 added a `title` property to route definitions that automatically sets `document.title` when the route is activated:

```typescript
const routes: Routes = [
  { path: 'home',    component: HomeComponent,    title: 'Home' },
  { path: 'profile', component: ProfileComponent, title: 'My Profile' },
];
```

You can also extend `TitleStrategy` to customize the format (e.g., append a brand name) or use a `ResolveFn` for dynamic titles based on route data.

---

## Intermediate Level Questions

---

### Q6. What is `FormBuilder.nonNullable` and when should you use it?

**Answer:**
`FormBuilder.nonNullable` creates form controls that **never return `null`**. By default, calling `.reset()` on a `FormControl` sets its value to `null`, but nonNullable controls reset to their **initial value** instead.

```typescript
const fb = inject(FormBuilder);

// Standard — value can be null after reset()
const standard = fb.control('hello');      // string | null

// NonNullable — value is always string, even after reset()
const strict = fb.nonNullable.control('hello');  // string (no null!)

// Whole group as nonNullable
const form = fb.nonNullable.group({
  username: [''],   // FormControl<string>
  password: ['']    // FormControl<string>
});
```

Use `nonNullable` for login/registration forms where you always need string values for submission.

---

### Q7. How does lazy loading work with standalone components in Angular 14?

**Answer:**
Angular 14 introduces two new lazy loading patterns:

**`loadComponent`** — lazy load a single standalone component:
```typescript
{
  path: 'settings',
  loadComponent: () =>
    import('./settings/settings.component').then(m => m.SettingsComponent)
}
```

**`loadChildren` with a routes array** — lazy load a feature's routes (replaces lazy-loaded NgModule):
```typescript
{
  path: 'admin',
  loadChildren: () =>
    import('./admin/admin.routes').then(m => m.ADMIN_ROUTES)
}
```

This is much simpler than before because there's no need for `AdminModule` with `RouterModule.forChild(routes)`.

---

### Q8. Can a standalone component be used inside an existing NgModule app?

**Answer:**
Yes. Standalone components are **interoperable** with NgModules in both directions:

**Using standalone component in NgModule:**
```typescript
@NgModule({
  imports: [
    BrowserModule,
    MyStandaloneComponent,   // ← import standalone component directly
  ],
  declarations: [AppComponent],
  bootstrap: [AppComponent]
})
export class AppModule {}
```

**Using NgModule-based component inside standalone component:**
```typescript
@Component({
  standalone: true,
  imports: [
    CommonModule,            // ← import CommonModule (NgModule) to get *ngIf, *ngFor
    MatButtonModule,         // ← Angular Material NgModule
    MyStandaloneComponent    // ← standalone component
  ]
})
export class MyPage {}
```

---

### Q9. What are Extended Developer Diagnostics in Angular 14?

**Answer:**
Extended Diagnostics are **additional compile-time checks** added by the Angular template compiler to catch common template mistakes that are technically valid but logically wrong.

Examples:
- **`nullishCoalescingNotNullable`**: warns when `??` is used on a value that is never null/undefined
- **`optionalChainNotNullable`**: warns when `?.` is used on a value that can't be null
- **`invalidBananaInBox`**: errors when `([value])` is used instead of `[(value)]`

Configure in `tsconfig.json` under `angularCompilerOptions.extendedDiagnostics.checks`. Each check can be set to `"warning"`, `"error"`, or `"suppress"`.

---

### Q10. What is the difference between `UntypedFormControl` and `FormControl` in Angular 14?

**Answer:**

| | `FormControl` (Angular 14+) | `UntypedFormControl` |
|---|---|---|
| Type | Generic `FormControl<T>` | Equivalent to old `FormControl` (returns `any`) |
| Value type | `T \| null` or `T` (nonNullable) | `any` |
| Purpose | New code — type-safe | Migration escape hatch — gradual adoption |
| Added in | Originally, now fully typed | Angular 14 (same API as old `FormControl`) |

`UntypedFormControl` was introduced specifically to allow teams to upgrade to Angular 14 **without rewriting all forms at once**. The plan is to gradually migrate from `UntypedFormControl` to typed `FormControl<T>` over time.

---

## Advanced Level Questions

---

### Q11. How does the Standalone Component Schematic migration work (`ng generate @angular/core:standalone`)?

**Answer:**
The schematic runs in three phases:

```bash
# Phase 1: Convert components/directives/pipes to standalone
ng generate @angular/core:standalone --mode=convert-to-standalone

# Phase 2: Remove unnecessary NgModules (those that only existed for declarations)
ng generate @angular/core:standalone --mode=prune-ng-modules

# Phase 3: Switch bootstrapModule() to bootstrapApplication()
ng generate @angular/core:standalone --mode=standalone-bootstrap
```

Each phase makes atomic changes that can be reviewed and tested independently. After all three phases, the app has no NgModules and uses `bootstrapApplication`.

**Limitations:** The schematic handles most cases automatically but may need manual review for:
- Complex `forRoot()`/`forChild()` patterns
- Dynamic module loading via `loadChildren`
- Non-standard `NgModule` patterns

---

### Q12. How does the `title` route property work with child routes?

**Answer:**
The `TitleStrategy.buildTitle()` method traverses the route tree and finds the **deepest activated route with a `title` defined**:

```typescript
const routes: Routes = [
  {
    path: 'products',
    title: 'Products',         // parent title
    children: [
      {
        path: '',
        component: ProductListComponent
        // no title → inherits from parent: "Products"
      },
      {
        path: ':id',
        component: ProductDetailComponent,
        title: productTitleResolver   // overrides parent title
        // → "Running Shoes | My Store"  (from resolver)
      }
    ]
  }
];
```

Child route titles **override** parent titles. If a child has no title, the parent's title is used.

---

### Q13. Explain the `FormControl` type hierarchy in Angular 14.

**Answer:**
Angular 14 introduces a type hierarchy for form controls:

```
AbstractControl<TValue, TRawValue>
├── FormControl<TValue>
├── FormGroup<TControls>
└── FormArray<TControl>
```

- `AbstractControl.value` — may be `null` if any child is disabled
- `AbstractControl.getRawValue()` — always returns full value including disabled controls

```typescript
const form = new FormGroup({
  name: new FormControl<string>('Alice', { nonNullable: true }),
  age:  new FormControl<number>(30)
});

// .value type — accounts for disabled controls
// { name: string; age: number | null }
form.value;

// .getRawValue() — complete, always includes disabled
// { name: string; age: number | null }
form.getRawValue();

// Access control type-safely
form.controls.name.value;   // type: string (nonNullable)
form.controls.age.value;    // type: number | null
```

---

### Q14. What is the difference between `provideRouter()` and `RouterModule.forRoot()` in Angular 14?

**Answer:**

| | `RouterModule.forRoot()` | `provideRouter()` |
|---|---|---|
| Approach | NgModule-based | Standalone/function-based |
| Usage | `imports` in NgModule | `providers` in `bootstrapApplication` |
| Features | Via `RouterModule.forRoot(routes, { preloadingStrategy })` | Via `withXxx()` functions |
| Tree-shaking | Less effective | Better tree-shaking |

```typescript
// Old — NgModule
@NgModule({
  imports: [RouterModule.forRoot(routes, { preloadingStrategy: PreloadAllModules })]
})
export class AppModule {}

// New — Standalone Angular 14
bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(
      routes,
      withPreloading(PreloadAllModules),
      withDebugTracing()
    )
  ]
});
```

`provideRouter` with feature functions is more tree-shakable — features you don't use aren't included in the bundle.

---

### Q15. How do you handle form validation errors with typed forms in Angular 14?

**Answer:**
```typescript
@Component({ standalone: true, imports: [ReactiveFormsModule, CommonModule] })
export class LoginComponent {
  form = new FormGroup({
    email:    new FormControl<string>('', {
      nonNullable: true,
      validators: [Validators.required, Validators.email]
    }),
    password: new FormControl<string>('', {
      nonNullable: true,
      validators: [Validators.required, Validators.minLength(8)]
    })
  });

  // Helper — fully typed getter
  get emailCtrl() { return this.form.controls.email; }
  get passwordCtrl() { return this.form.controls.password; }

  get emailErrors(): string[] {
    const errors = this.emailCtrl.errors;
    if (!errors || !this.emailCtrl.touched) return [];

    const messages: string[] = [];
    if (errors['required'])  messages.push('Email is required');
    if (errors['email'])     messages.push('Must be a valid email address');
    return messages;
  }
}
```

```html
<input [formControl]="emailCtrl" type="email">
<ul *ngIf="emailErrors.length">
  <li *ngFor="let msg of emailErrors">{{ msg }}</li>
</ul>
```

---

## Scenario-Based Questions

---

### Q16. Your team is migrating a large Angular 13 app to Angular 14. There are 150 components with NgModule. How would you approach the migration?

**Answer:**
**Strategy: Incremental Migration — don't migrate everything at once.**

```bash
# Step 1: Upgrade to Angular 14
ng update @angular/core@14 @angular/cli@14

# Step 2: Run typed forms migration (automated)
ng generate @angular/core:typed-forms
# This converts FormControl → UntypedFormControl (backward compatible)
# Then gradually replace UntypedFormControl with typed FormControl<T>

# Step 3: Convert components to standalone gradually
# Start with leaf components (no children), then work upward
ng generate @angular/core:standalone --mode=convert-to-standalone

# Step 4: After most components are standalone, prune NgModules
ng generate @angular/core:standalone --mode=prune-ng-modules

# Step 5: Finally switch to bootstrapApplication
ng generate @angular/core:standalone --mode=standalone-bootstrap
```

**Priority order for migration:**
1. Leaf components (buttons, inputs, cards) — no dependencies
2. Feature components that use leaves
3. Page-level components
4. Root `AppComponent` and `AppModule` last

---

### Q17. A junior developer asks: "If standalone components don't need NgModule, why does Angular still have NgModule?" How would you answer?

**Answer:**
NgModules still exist for three reasons:

1. **Backward compatibility**: Millions of Angular apps use NgModules. Removing them would be a breaking change.

2. **Third-party libraries**: Many libraries (NgRx, Angular Material, ngx-translate) still use NgModule APIs. `importProvidersFrom` bridges them to standalone apps.

3. **Grouping providers**: `NgModule.forRoot()`/`forChild()` patterns are used to group and configure sets of providers. Standalone apps replace this with `provideXxx()` functions.

**In Angular 14 specifically**, NgModules are **optional** — you can write a full app without any NgModule. They will likely be deprecated eventually, but there's no timeline announced yet.

---

### Q18. How do you write unit tests for a standalone component?

**Answer:**
Standalone components are **easier to test** because you explicitly declare what they need:

```typescript
// card.component.spec.ts
import { ComponentFixture, TestBed } from '@angular/core/testing';
import { CardComponent } from './card.component';
import { By } from '@angular/platform-browser';

describe('CardComponent', () => {
  let fixture: ComponentFixture<CardComponent>;
  let component: CardComponent;

  beforeEach(async () => {
    await TestBed.configureTestingModule({
      // For standalone components — use imports instead of declarations!
      imports: [CardComponent]
    }).compileComponents();

    fixture = TestBed.createComponent(CardComponent);
    component = fixture.componentInstance;
    component.title = 'Test Title';
    fixture.detectChanges();
  });

  it('should display the title', () => {
    const h2 = fixture.debugElement.query(By.css('h2'));
    expect(h2.nativeElement.textContent).toBe('Test Title');
  });
});

// Override imports for testing (e.g., use mock service)
await TestBed.configureTestingModule({
  imports: [CardComponent]
}).overrideComponent(CardComponent, {
  set: {
    imports: [MockCommonModule]   // replace CommonModule with mock
  }
}).compileComponents();
```

---

### Q19. You notice form values have type `any` even after upgrading to Angular 14. What could be the issue?

**Answer:**
The most common reasons:

1. **Using `UntypedFormControl`**: The migration schematic converts old `FormControl` to `UntypedFormControl`. You need to manually replace them with typed `FormControl<T>`:
   ```typescript
   // Fix: replace UntypedFormControl with typed version
   new UntypedFormControl('')  →  new FormControl<string>('', { nonNullable: true })
   ```

2. **Not providing generic type**: `new FormControl('')` without a type annotation allows TypeScript to infer `string | null`. Without `{ nonNullable: true }`, calling `.reset()` makes it `null`.

3. **Using `.get()` instead of `.controls`**: `form.get('name')` returns `AbstractControl | null`. Use `form.controls.name` for a typed reference.

---

### Q20. What is the difference between `loadComponent` and `loadChildren` in Angular 14 lazy routing?

**Answer:**

| | `loadComponent` | `loadChildren` |
|---|---|---|
| Loads | A single standalone component | A set of routes (routes array) |
| Replaces | Lazy NgModule with one component | Lazy NgModule with `RouterModule.forChild()` |
| When to use | Simple pages, single component routes | Feature areas with multiple routes |

```typescript
const routes: Routes = [
  // loadComponent — single page
  {
    path: 'terms',
    loadComponent: () =>
      import('./terms/terms.component').then(m => m.TermsComponent)
  },

  // loadChildren — feature area (multiple routes inside)
  {
    path: 'shop',
    loadChildren: () =>
      import('./shop/shop.routes').then(m => m.SHOP_ROUTES)
    // shop.routes.ts exports:
    // export const SHOP_ROUTES: Routes = [
    //   { path: '', component: ShopListComponent },
    //   { path: 'cart', component: CartComponent },
    //   { path: 'product/:id', component: ProductDetailComponent }
    // ]
  }
];
```

---

## Quick Reference Card

### Angular 14 Key Facts for Interviews

| Question | Answer |
|---|---|
| Release date | June 2, 2022 |
| Biggest feature | Standalone Components (developer preview) |
| Forms change | Typed Reactive Forms — `FormControl<T>` |
| Bootstrap function | `bootstrapApplication()` replaces `bootstrapModule()` |
| Lazy load component | `loadComponent: () => import(...).then(m => m.Comp)` |
| Lazy load routes | `loadChildren: () => import(...).then(m => m.ROUTES)` |
| Page title | `title` property in route config or `TitleStrategy` |
| Form escape hatch | `UntypedFormControl` for gradual migration |
| NgModule bridge | `importProvidersFrom()` |
| Diagnostics | `extendedDiagnostics` in `angularCompilerOptions` |
| TypeScript version | 4.6.x |
| Optional injectors | `createComponent({ injector })` for scoped DI |

### Before vs After Cheatsheet

```
NgModule declarations   →  standalone: true + imports: []
bootstrapModule()       →  bootstrapApplication()
RouterModule.forRoot()  →  provideRouter()
HttpClientModule        →  provideHttpClient()
FormControl (any)       →  FormControl<string> / FormControl<number>
loadChildren: Module    →  loadChildren: routes array
                        →  loadComponent: single component
SharedModule import     →  import each standalone component directly
entryComponents         →  (still not needed — removed in v13)
```